### PageIndex — Vectorless RAG

#### Key Concept
Traditional RAG → chunk → embed → cosine similarity → retrieve 

PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections


-------------------------------------------------------------------------------------------------------------------------------------------

### The problem with vector RAG:
##### Similarity ≠ Relevance
A chunk about "market conditions" may score higher than the actual answer section just because it shares more words with your query.

### Section 1: Install & Setup
What we do here:

Install PageIndex SDK + gemini / openai

Load API keys from .env

Initialize both clients

🔑 Get your PageIndex API key from: https://dash.pageindex.ai/api-keys

🔑 Get your OpenAI API key from: https://platform.openai.com

In [1]:
# Install required packages
!pip install -U pageindex  python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# load API keys from .env file
import os
from dotenv import load_dotenv

load_dotenv()

# get Gemini API key
gemini_api_key = os.getenv("GOOGLE_API_KEY")
pageindex_api_key = os.getenv("PAGEINDEX_API_KEY")

if gemini_api_key:
    masked_key = gemini_api_key[:4] + "..." + gemini_api_key[-4:] if len(gemini_api_key) > 8 else "****"
    print(f"Gemini API key loaded successfully: {masked_key}")
else:
    print("Failed to load Gemini API key. Please check your .env file.")


if pageindex_api_key:
    masked_key  = pageindex_api_key[:4] + "..." + pageindex_api_key[-4:] if len(pageindex_api_key) > 8 else "****"
    print(f"PageIndex API key loaded successfully: {masked_key}")
else:
    print("Failed to load PageIndex API key. Please check your .env file.")

Gemini API key loaded successfully: AIza...tYzA
PageIndex API key loaded successfully: 57e2...ede3


In [11]:
from pageindex import PageIndexClient
from langchain_google_genai import ChatGoogleGenerativeAI

pi_client     = PageIndexClient(api_key=pageindex_api_key)
gemini_client = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_api_key, model_kwargs={"response_mime_type": "application/json"})

print("Clients initialized successfully!")

c:\Users\BHARATH\github\VectorlessRAG\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: UserWarning: Parameters {'response_mime_type'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  exec(code_obj, self.user_global_ns, self.user_ns)


Clients initialized successfully!


### Section 2: Upload & Index a PDF

What happens here:

1.Upload your PDF to the PageIndex cloud.

2.PageIndex uses an LLM to read the document structure.

3.Builds a hierarchical tree index (like a smart Table of Contents).

4.Returns a doc_id for all future operations.

Why NO chunking?

Instead of cutting the document into arbitrary 500-token pieces, PageIndex respects the document's natural section boundaries — chapters, sub-sections, paragraphs — as the author intended.

In [4]:
# ── Upload your PDF ─────────────────────────────────────────────────────────
# Replace with the path to your PDF file
# Great candidates: Annual reports, research papers, legal docs, textbooks

PDF_PATH = "data/llm.pdf"   # ← change this

print(f"📤 Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print(f"✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")
print("   (Save this ID — you'll use it throughout the notebook)")

📤 Uploading: data/llm.pdf
✅ Uploaded!
📋 Document ID: pi-cmojnjybd00ik01qw68v0ld6j
   (Save this ID — you'll use it throughout the notebook)


In [5]:
# ── Poll until processing is complete ───────────────────────────────────────
# PageIndex builds the tree asynchronously.
# For a 50-page PDF this typically takes 30–90 seconds.
import os, json, time
print("⏳ Building tree index...")
print("   (This runs once per document — the index is cached for reuse)")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")
    
    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break
    
    time.sleep(5)

⏳ Building tree index...
   (This runs once per document — the index is cached for reuse)
   Status: completed

✅ Tree index ready!


#### Section 3: Inspect the Tree Structure
What the tree looks like:
```text
Document
├── Introduction (pages 1-3)
│   └── Background (pages 1-2)
├── Financial Stability (pages 21-31)
│   ├── Monitoring Vulnerabilities (pages 22-28)
│   └── International Cooperation (pages 28-31)
└── Conclusion (pages 45-47)
```

Each node has:

* node_id — unique ID used during retrieval
* title — section heading
* page_index — page number in original PDF
* text — section summary (when node_summary=True)
* nodes — child sections (nested)
* This structure is what the LLM reasons over during retrieval.

In [6]:
# ── Fetch the full tree ─────────────────────────────────────────────────────
tree_result  = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"📊 Top-level sections: {len(pageindex_tree)}")
print("\n🌲 Raw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

📊 Top-level sections: 5

🌲 Raw tree (first node):
{
  "title": "Introduction",
  "node_id": "0000",
  "page_index": 1,
  "summary": "This text introduces Large Language Models (LLMs) as a significant advancement in AI, defining them as deep neural networks trained on vast text data capable of processing, understanding, and generating human-like text for various tasks. The document outlines its coverage of LLM architecture timelines, fine-tuning techniques, efficient training methods, inference acceleration, and practical applications with code examples.",
  "text": "# Introduction\n\nThe advent of Large Language Models (LLMs) represents a seismic shift in the world of artificial intelligence. Their ability to process, generate, and understand user intent is fundamentally changing the way we interact with information and technology.\n\nAn LLM is an advanced artificial intelligence system that specializes in processing, understanding, and generating human-like text. These systems are typ

In [7]:
# ── Pretty-print the full tree ───────────────────────────────────────────────
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("📚 Full Document Structure:\n")
print_tree(pageindex_tree)

📚 Full Document Structure:

[0000] Introduction  (p.1)
[0001] Why language models are important  (p.2)
[0002] Large language models  (p.3)
  └─ [0003] Transformer  (p.4)
    └─ [0004] Input preparation and embedding  (p.5)
    └─ [0005] Multi-head attention  (p.6)
    └─ [0006] Understanding self-attention  (p.6)
    └─ [0007] Multi-head attention: power in diversity  (p.8)
    └─ [0008] Layer normalization and residual connections  (p.9)
    └─ [0009] Feedforward layer  (p.9)
    └─ [0010] Encoder and decoder  (p.10)
    └─ [0011] Mixture of Experts (MoE)  (p.11)
[0012] The evolution of transformers  (p.13)
  └─ [0013] GPT-1  (p.13)
  └─ [0014] BERT  (p.15)
  └─ [0015] GPT-2  (p.15)
  └─ [0016] Supervised fine-tuning  (p.16)
[0017] Accelerating inference  (p.17)
  └─ [0018] Trade offs  (p.18)
    └─ [0019] The Quality vs Latency/Cost Tradeoff  (p.18)
    └─ [0020] The Latency vs Cost Tradeoff  (p.19)
  └─ [0021] Output-approximating methods  (p.20)


In [8]:
# ── Count total nodes ────────────────────────────────────────────────────────
def count_nodes(nodes):
    total = len(nodes)
    for n in nodes:
        if n.get("nodes"):
            total += count_nodes(n["nodes"])
    return total

total = count_nodes(pageindex_tree)
print(f"🔢 Total nodes in tree: {total}")
print("   Each node = one retrievable section of the document")

🔢 Total nodes in tree: 22
   Each node = one retrievable section of the document


### Section 4: LLM Tree Search — The Core of PageIndex
This is where PageIndex fundamentally differs from vector RAG.

```text
Vector RAG retrieval:
query → embed → cosine_similarity(query_vec, all_chunk_vecs) → top-k chunks
```

Problem: finds what's similar, not what's relevant

```text
PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [18]:
# ── LLM Tree Search Function ─────────────────────────────────────────────────
def llm_tree_search(query: str, tree: list) -> dict:
    """
    Core PageIndex retrieval:
    Sends the query + document tree to an LLM.
    LLM reasons over the structure and returns relevant node_ids.
    
    Returns: dict with 'thinking' (reasoning) and 'node_list' (node IDs)
    """
    
    # Compress tree to save tokens — only send titles + short summaries
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]  # first 150 chars
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out
    
    compressed_tree = compress(tree)
    
    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = gemini_client.invoke(
        prompt
    )
    #print("\n🤖 LLM Response:", response)
    return json.loads(response.content)

In [19]:
# ── Test with a sample query ─────────────────────────────────────────────────
query = "What is the Multi-head attention?"

print(f"🔍 Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("🧠 LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("🎯 Selected Node IDs:", result.get("node_list", []))

🔍 Query: What is the Multi-head attention?

🧠 LLM Reasoning:
The query asks for a definition of 'Multi-head attention'. I will look for nodes with 'Multi-head attention' in their titles or summaries. Node '0005' is titled 'Multi-head attention' and its summary indicates it introduces the module. Node '0007' is titled 'Multi-head attention: power in diversity' and its summary provides a detailed explanation of how it works. Both are highly relevant and directly answer the query.

🎯 Selected Node IDs: ['0005', '0007']


### Section 5: Full End-to-End RAG Pipeline
3 steps:

1.Tree Search → LLM picks relevant node_ids

2.Retrieve → Fetch the actual section content from those nodes

3.Generate → LLM writes a grounded answer with page citations

What makes this better than vector RAG:

* Retrieved content has titles + page numbers (traceable)
* LLM can cite exactly which section the answer comes from
* No hallucination from irrelevant chunks

In [20]:
# ── Helper: Find nodes by ID ─────────────────────────────────────────────────

def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [21]:
# ── Generate answer from retrieved nodes ─────────────────────────────────────

def generate_answer(query: str, nodes: list) -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "⚠️ No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""
    
    response = gemini_client.invoke(
        prompt
    )
    
    return response.content

In [22]:
# ── The complete Vectorless RAG function ─────────────────────────────────────

def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"🔍 Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result  = llm_tree_search(query, tree)
    node_ids       = search_result.get("node_list", [])
    
    if verbose:
        print(f"\n🧠 Reasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"🎯 Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"📄 Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\n📝 Answer:\n{answer}")
    
    return answer

In [23]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is the Multi-head attention?",
    tree=pageindex_tree
)

🔍 Query: What is the Multi-head attention?

🧠 Reasoning: The query asks for a definition of 'Multi-head attention'. I will scan the document titles and summaries for exact matches or highly relevant sections. Node '0005' is titled 'Multi-head attention' and...
🎯 Retrieved node IDs: ['0005', '0007']
📄 Sections found: ['Multi-head attention', 'Multi-head attention: power in diversity']

📝 Answer:
{
  "answer": "Multi-head attention is a module that processes input token embedding vectors (Multi-head attention, Page 6). It utilizes multiple parallel sets of Q, K, V weight matrices, with each 'head' focusing on different aspects of input relationships. The outputs from these heads are concatenated and linearly transformed, providing the model with a richer representation of the input sequence (Multi-head attention: power in diversity, Page 8). This mechanism enhances the model's ability to manage complex language patterns and long-range dependencies by considering multiple interpretations 

In [25]:
# ── Run the full pipeline ────────────────────────────────────────────────────
answer = vectorless_rag(
    query="What is the Latency vs Cost Tradeoff?",
    tree=pageindex_tree
)

🔍 Query: What is the Latency vs Cost Tradeoff?

🧠 Reasoning: The query is 'What is the Latency vs Cost Tradeoff?'. I need to find nodes whose titles or summaries directly address this concept. I will scan the document tree for keywords like 'Latency', 'Cost', a...
🎯 Retrieved node IDs: ['0020']
📄 Sections found: ['The Latency vs Cost Tradeoff']

📝 Answer:
{
  "answer": "The Latency vs Cost Tradeoff, also known as the latency vs throughput tradeoff, refers to the balance between the speed of an LLM inference and its associated expense. Better throughput, which is a system's ability to handle multiple requests efficiently, directly reduces LLM inference cost. This tradeoff is crucial because LLM inference is typically the slowest and most expensive part of the stack, requiring intentional balancing to tailor performance to specific product needs or use cases. For instance, bulk inference prioritizes cost over latency, while an LLM chatbot product places higher importance on request laten